In [1]:
import os

# Ensure the notebook executes from the project root, even if located in the 'code' folder
if os.path.basename(os.getcwd()).lower() == "code":
    os.chdir("..")

print("Current Working Directory:", os.getcwd())

DATA_PATH = "./data/"
MODEL_DIR = "./models/"
PRED_DIR = "./predictions/"

# Ensure output directories exist without manual intervention
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)

Current Working Directory: c:\Users\Ekansh\Downloads\Bootcamp_check\25b0651_cv_bootcamp


In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import confusion_matrix
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# 2. Load Data
labels_df = pd.read_csv("data/labels.csv")
train_idx = np.loadtxt("data/train_idx.txt", dtype=int)
val_idx = np.loadtxt("data/val_idx.txt", dtype=int)
with open("data/classes.txt", "r") as f:
    classes = [line.strip() for line in f.readlines()]

# Load images into memory (requires ~380MB RAM, safe for Colab)
train_images_all = np.load("data/train_images.npy")
test_images = np.load("data/test_images.npy")

print(f"Loaded {len(train_idx)} train, {len(val_idx)} val, {len(test_images)} test images.")



In [ ]:
# Isolate training labels
train_labels = labels_df.iloc[train_idx]
clean_train = train_labels['clean_label'].values
noisy_train = train_labels['noisy_label'].values

# Calculate Noise Rate
noise_rate = np.mean(clean_train != noisy_train) #calculates the percentage or fraction of elements that do not match between two NumPy arrays
noise_percentage = noise_rate * 100

print(f"Task 1 - Noise Rate: {noise_percentage:.4f}%")

# Compute Confusion Matrix
# Rows = True (Clean), Columns = Predicted (Noisy)
conf_matrix= confusion_matrix(clean_train, noisy_train)
 #The rows represent the True/Clean labels, and the columns represent the Noisy/Human labels
with open("data/classes.txt", "r") as f:
    classes = [line.strip() for line in f.readlines()]

# Wrap the matrix in a Pandas DataFrame
conf_matrix_df = pd.DataFrame(conf_matrix, index=classes, columns=classes)

# Name the axes for absolute clarity
conf_matrix_df.index.name = 'True (Clean)'
conf_matrix_df.columns.name = 'Human (Noisy)'

print("Task 1 - Confusion Matrix (Clean vs Noisy):") #diagonal are all correctly mappped labels
# print(conf_matrix_df)
display(conf_matrix_df) #if row 3 is "cat" and column 5 is "dog", a high number in that specific cell indicates that human annotators consistently mistook cats for dogs.

# Initialize results JSON
results = {
    "name": "Your Name",
    "roll_no": "Your Roll No",
    "seed": 42,
    "training_setup": {
        "optimizer": "SGD",
        "learning_rate_schedule": "CosineAnnealingLR",
        "batch_size": 128,
        "epochs": 60,
        "augmentation": "RandomCrop + RandomHorizontalFlip", #anything else like hue or saturation would unnecessarily scramble the data and adding more
        "gpu_used": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
    },
    "task1": {
        "noise_rate": float(noise_percentage),
        "val_acc_noisy_labels__clean_ref_model": None,
        "val_acc_noisy_labels__noisy_ce_model": None
    },
    "task2": {
    "memorization_start_epoch": None,
    "early_stop_epoch": None,
    "val_acc_noisy_labels_at_early_stop": None
  },
  "task3": {
    "method": ""
  },
  "task4": {
    "methods": [
      {"name": "", "file": "t4_<name>.csv", "val_acc_noisy_labels": None}
    ],
    "best_method": ""
  },
  "task5": {
    "val_acc_synthetic_labels__baseline": None,
    "val_acc_synthetic_labels__best_method": None
}}

Task 1 - Noise Rate: 40.2422%
Task 1 - Confusion Matrix (Clean vs Noisy):


Human (Noisy),airplane,automobile,bird,cat,deer,dog,frog,horse,ship,truck
True (Clean),,,,,,,,,,
airplane,2925,293,314,89,94,105,85,75,379,146
automobile,191,2657,63,116,80,68,60,64,71,1126
bird,263,165,2842,212,254,216,222,147,106,77
cat,180,160,354,2168,194,924,206,106,91,86
deer,184,142,316,184,2195,491,174,683,94,50
dog,150,131,240,834,183,2620,123,136,55,61
frog,170,166,373,350,229,257,2676,124,96,62
horse,179,120,111,100,233,289,101,3246,77,50
ship,369,333,94,65,96,79,71,99,3076,214


In [ ]:
class CIFAR10NDataset(Dataset):
    def __init__(self, images, labels_df, indices, target_col='noisy_label', transform=None):
        self.images = images[indices] #from the array shape (50000, 32, 32, 3)).
        self.indices = indices # makes it global ref
        self.target_labels = labels_df.iloc[indices][target_col].values #Extract target_column labels for training target

        # Keep clean/noisy for Task 2 tracking (metrics only, NEVER for training)
        self.clean_labels = labels_df.iloc[indices]['clean_label'].values
        self.noisy_labels = labels_df.iloc[indices]['noisy_label'].values

        self.transform = transform #Store transform pipeline

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        img = Image.fromarray(self.images[i])
        if self.transform: #Executes the transformation pipeline on the image: applies data augmentation, converts pixels to floats between $0.0$ and $1.0$, and normalizes the color channels. The output img is now a PyTorch Tensor of shape (3, 32, 32).
            img = self.transform(img)

        target = torch.tensor(self.target_labels[i], dtype=torch.long) #converting all of the variables to long tensors to fit format
        clean = torch.tensor(self.clean_labels[i], dtype=torch.long)
        noisy = torch.tensor(self.noisy_labels[i], dtype=torch.long)
        global_idx = self.indices[i]

        return global_idx, img, target, clean, noisy

# Standard CIFAR augmentations
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4), #Pads the $32 \times 32$ image by 4 pixels on all sides (making it $40 \times 40$),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), #converts to CHW
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
  #normalise is value - mean / SD
    #(0.4914, 0.4822, 0.4465) and (0.2023, 0.1994, 0.2010) are the empirical per-channel RGB mean and standard deviation values of the standard CIFAR-10 training set.
])

transform_eval = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Create DataLoaders
# 1. Clean Reference DataLoader (Uses clean_label for targets)
train_ds_clean = CIFAR10NDataset(train_images_all, labels_df, train_idx, 'clean_label', transform_train)
train_loader_clean = DataLoader(train_ds_clean, batch_size=128, shuffle=True, num_workers=2) #: num: Uses 2 parallel background CPU processes to fetch, decode, and transform images

# 2. Noisy CE DataLoader (Uses noisy_label for targets)
train_ds_noisy = CIFAR10NDataset(train_images_all, labels_df, train_idx, 'noisy_label', transform_train)
train_loader_noisy = DataLoader(train_ds_noisy, batch_size=128, shuffle=True, num_workers=2)

# 3. Validation DataLoader (No augmentation)
val_ds = CIFAR10NDataset(train_images_all, labels_df, val_idx, 'noisy_label', transform_eval)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=2)
#batch_size=256: Can be larger than the training batch size because validation does not compute gradients

In [ ]:
from model import ResNet18 # Ensure model.py is uploaded to your Colab environment

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def train_baseline(train_loader, val_loader, model_name="noisy_ce", epochs=60): #train_loader feeds data
    net = ResNet18(num_classes=10).to(device) #Initializes the neural network and ships it to the GPU/
    criterion = nn.CrossEntropyLoss() #for calculating the error and feed it backward
    optimizer = optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4) #Stochastic gradient descent , grad is used to find optimal direction of path
    #lr multiplied to grad in weights formula to show effect of one to another , momentum pushes against Local min, weight decay decays the parameters to reduce their sensitivity and rigidness
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs) #is solely responsible for the smooth, gradual reduction of the learning rate along a cosine curve
    #as it reduces later on Reducing the learning rate down to a tiny fraction forces the model to take microscopic steps,

    dynamics_log = []

    for epoch in range(epochs):
        net.train()
        correct_clean_subset, total_clean_subset = 0, 0
        correct_mislabeled_subset, total_mislabeled_subset = 0, 0

        for idx, inputs, targets, cleans, noisys in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            cleans, noisys = cleans.to(device), noisys.to(device)

            optimizer.zero_grad() #prevents addition of gradients in next batch , prevents new gradient to have old gradient in it , new gradient calculated newly (not related to weights)
            outputs = net(inputs) #we get probabilities for the 10 classes
            loss = criterion(outputs, targets)
            loss.backward() #cross entropy loss, calculation of weights
            optimizer.step() #adjust weights

            # --- Task 2: Tracking Dynamics ---
            _, predicted = outputs.max(1) #array of outputs

            # Mask for images where human annotation was correct
            is_clean_mask = (cleans == noisys) #Creates a True/False array. If the clean label equals the noisy human label, it's True
            is_mislabeled_mask = ~is_clean_mask

            # Accuracy on correctly labeled subset
            correct_clean_subset += predicted[is_clean_mask].eq(cleans[is_clean_mask]).sum().item() #sum of those matching outputs (eq=equal)
            total_clean_subset += is_clean_mask.sum().item() #adds all the images within the given clean subset

            # Accuracy on mislabeled subset
            correct_mislabeled_subset += predicted[is_mislabeled_mask].eq(cleans[is_mislabeled_mask]).sum().item()
            total_mislabeled_subset += is_mislabeled_mask.sum().item()

        scheduler.step() #lr reduces

        # --- Validation Tracking ---
        net.eval()
        val_noisy_correct, val_clean_correct, val_total = 0, 0, 0
        with torch.no_grad(): #stop tracking gradient
            for idx, inputs, _, cleans, noisys in val_loader:
                inputs, cleans, noisys = inputs.to(device), cleans.to(device), noisys.to(device)
                outputs = net(inputs)
                _, predicted = outputs.max(1)

                val_noisy_correct += predicted.eq(noisys).sum().item()
                val_clean_correct += predicted.eq(cleans).sum().item() #list of clean labels whuch are there in it
                val_total += inputs.size(0)

        # Calculate Epoch Metrics
        train_acc_correct = (correct_clean_subset / total_clean_subset) * 100
        train_acc_mislabeled = (correct_mislabeled_subset / total_mislabeled_subset) * 100
        val_acc_noisy = (val_noisy_correct / val_total) * 100
        val_acc_clean = (val_clean_correct / val_total) * 100 #total clean labels

        dynamics_log.append([epoch+1, train_acc_correct, train_acc_mislabeled, val_acc_noisy, val_acc_clean])
        print(f"Epoch {epoch+1:02d} | Train Acc (Clean/Mislabeled): {train_acc_correct:.1f}% / {train_acc_mislabeled:.1f}% | Val Acc (Noisy/Clean): {val_acc_noisy:.1f}% / {val_acc_clean:.1f}%")

    # If this is the noisy model, save the dynamics for Task 2
    if model_name == "noisy_ce":
        dyn_df = pd.DataFrame(dynamics_log, columns=["epoch", "train_acc_correct", "train_acc_mislabeled", "val_acc_noisy", "val_acc_clean"])
        dyn_df.to_csv("predictions/t2_dynamics.csv", index=False)

    return net, val_acc_noisy

import os

# --- 1. Clean Reference Model ---
clean_model_path = "models/clean_ref_model.pth"
model_clean_ref = ResNet18(num_classes=10).to(device)

if os.path.exists(clean_model_path):
    print("Found saved Clean Reference Model! Loading weights...")
    # Load the weights from Google Drive directly into the model
    model_clean_ref.load_state_dict(torch.load(clean_model_path, map_location=device))
else:
    print("No saved model found. Training Clean Reference Model from scratch...")
    torch.manual_seed(42)
    model_clean_ref, val_acc_noisy_clean_ref = train_baseline(train_loader_clean, val_loader, model_name="clean_ref")

    # Save the weights to Google Drive immediately after training
    torch.save(model_clean_ref.state_dict(), clean_model_path)
    print("Clean Reference Model saved to Drive.")


# --- 2. Noisy CE Model ---
noisy_model_path = "models/noisy_ce_model.pth"
model_noisy_ce = ResNet18(num_classes=10).to(device)

if os.path.exists(noisy_model_path):
    print("\nFound saved Noisy CE Model! Loading weights...")
    model_noisy_ce.load_state_dict(torch.load(noisy_model_path, map_location=device))
else:
    print("\nNo saved model found. Training Noisy CE Model from scratch...")
    torch.manual_seed(42)
    model_noisy_ce, val_acc_noisy_noisy_ce = train_baseline(train_loader_noisy, val_loader, model_name="noisy_ce")

    torch.save(model_noisy_ce.state_dict(), noisy_model_path)
    print("Noisy CE Model saved to Drive.")

# Update Task 1 Results JSON
results["task1"]["val_acc_noisy_labels__clean_ref_model"] = val_acc_noisy_clean_ref
results["task1"]["val_acc_noisy_labels__noisy_ce_model"] = val_acc_noisy_noisy_ce

In [ ]:
class CIFAR10TestDataset(Dataset):
    def __init__(self, images, transform=None):
        self.images = images
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.fromarray(self.images[idx])
        if self.transform:
            img = self.transform(img)
        return idx, img

test_ds = CIFAR10TestDataset(test_images, transform_eval)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

def generate_predictions(model, loader, filename):
    model.eval()
    preds_list = []
    with torch.no_grad():
        for ids, inputs in loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)

            for img_id, pred in zip(ids, predicted):
                preds_list.append(f"{img_id.item()},{pred.item()}\n")

    with open(f"predictions/{filename}", 'w') as f:
        f.write("id,pred\n")
        f.writelines(preds_list)


In [ ]:
# Generate Task 1 prediction files
generate_predictions(model_clean_ref, test_loader, "t1_clean_ref.csv")
generate_predictions(model_noisy_ce, test_loader, "t1_noisy_ce.csv")

# Save final results.json
with open("results.json", "w") as f:
    json.dump(results, f, indent=4)
print("Task 1 complete. Predictions and JSON saved.")

In [ ]:
import os
import pandas as pd
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from model import ResNet18 # Ensure model.py is accessible

device = 'cuda' if torch.cuda.is_available() else 'cpu'
os.makedirs("models", exist_ok=True)

# --- 1. Analyze Task 2 Dynamics CSV ---
# This requires t2_dynamics.csv to already exist in your predictions folder
dyn_df = pd.read_csv('predictions/t2_dynamics.csv') #is a 2D table

best_epoch_idx = dyn_df['val_acc_noisy'].idxmax() #gives biggest value in column ""
early_stop_epoch = int(dyn_df.loc[best_epoch_idx, 'epoch'])
val_acc_at_early_stop = float(dyn_df.loc[best_epoch_idx, 'val_acc_noisy'])

best_mislabeled_idx = dyn_df['train_acc_mislabeled'].idxmax()
memorization_start_epoch = int(dyn_df.loc[best_mislabeled_idx, 'epoch'])

print(f"Memorization Start Epoch determined as: Epoch {memorization_start_epoch}")
print(f"Early Stop Epoch determined as: Epoch {early_stop_epoch}")

# --- 2. Update results.json ---
with open('results.json', 'r') as f:
    results = json.load(f)

results["task2"]["memorization_start_epoch"] = memorization_start_epoch #train_acc_mislabeled tracks how often the network's prediction matches the hidden clean label for images that were given a wrong human label.
#eventually, the network abandons its general rules and uses its capacity to memorize that specific arrangement of pixels as a "dog"
results["task2"]["early_stop_epoch"] = early_stop_epoch #For those 13 epochs, the slight gains on the clean majority outweigh the slow rot of memorizing the noise
results["task2"]["val_acc_noisy_labels_at_early_stop"] = val_acc_at_early_stop
results["task3"]["method"] = "Cross-Entropy Loss at Early Stopping Epoch"

with open('results.json', 'w') as f:
    json.dump(results, f, indent=4)

# --- 3. Smart Train/Load Early Stop Model ---
early_model_path = f"models/early_stop_model_ep{early_stop_epoch}.pth"
early_stop_net = ResNet18(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()

if os.path.exists(early_model_path):
    print(f"Found saved Early Stop Model! Loading weights from epoch {early_stop_epoch}...")
    early_stop_net.load_state_dict(torch.load(early_model_path, map_location=device))
else:
    print(f"No saved model found. Training up to Epoch {early_stop_epoch}...")
    torch.manual_seed(42)
    optimizer = optim.SGD(early_stop_net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    # T_max MUST remain 60 to perfectly match the baseline learning rate curve as the original 1-60 epochs
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=60) #

    for epoch in range(early_stop_epoch):
        early_stop_net.train()
        for idx, inputs, targets, cleans, noisys in train_loader_noisy:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = early_stop_net(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
        scheduler.step()
        print(f"Completed Epoch {epoch+1}/{early_stop_epoch}")

    torch.save(early_stop_net.state_dict(), early_model_path)
    print("Early Stop Model saved to Drive.")

# --- 4. Generate Task 2 Early Stop Predictions ---
# (Requires test_loader to be defined)
def generate_test_predictions(model, loader, filename):
    model.eval()
    preds_list = []
    with torch.no_grad():
        for ids, inputs in loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            for img_id, pred in zip(ids, predicted):
                preds_list.append(f"{img_id.item()},{pred.item()}\n") #.item converts the singular object from tensor to raw no.

    with open(f"predictions/{filename}", 'w') as f:
        f.write("id,pred\n")
        f.writelines(preds_list) #appended csv lines 12th grd lmao

print("Generating Task 2 predictions...")
generate_test_predictions(early_stop_net, test_loader, "t2_early_stop.csv")

# --- 5. Generate Task 3 Noise Scores ---
# Requires a training loader without augmentations (transform_eval)
train_ds_eval = CIFAR10NDataset(train_images_all, labels_df, train_idx, 'noisy_label', transform_eval)
train_loader_score = DataLoader(train_ds_eval, batch_size=256, shuffle=False)

def generate_noise_scores(model, loader, filename):
    model.eval()
    score_criterion = nn.CrossEntropyLoss(reduction='none')
    scores_list = []

    with torch.no_grad():
        for global_idx, inputs, targets, _, _ in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            img_losses = score_criterion(outputs, targets)

            for idx, loss_val in zip(global_idx, img_losses):
                scores_list.append(f"{idx.item()},{loss_val.item():.4f}\n")

    with open(f"predictions/{filename}", 'w') as f:
        f.write("index,score\n")
        f.writelines(scores_list)

print("Generating Task 3 Noise Scores...")
generate_noise_scores(early_stop_net, train_loader_score, "t3_noise_scores.csv")
print("Tasks 2 and 3 complete!")

Memorization Start Epoch determined as: Epoch 33
Early Stop Epoch determined as: Epoch 46
Found saved Early Stop Model! Loading weights from epoch 46...
Generating Task 2 predictions...
Generating Task 3 Noise Scores...
Tasks 2 and 3 complete!


In [ ]:
import os
import json
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from model import ResNet18

os.makedirs("models", exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def train_label_smoothing(train_loader, val_loader, epochs=60):
    net = ResNet18(num_classes=10).to(device)
    # THIS is the Task 4 method: Label Smoothing!
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    optimizer = optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    dynamics_log = []

    for epoch in range(epochs):
        net.train()
        correct_clean_subset, total_clean_subset = 0, 0
        correct_mislabeled_subset, total_mislabeled_subset = 0, 0

        # Include cleans and noisys in the loop extraction
        for idx, inputs, targets, cleans, noisys in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            cleans, noisys = cleans.to(device), noisys.to(device)

            optimizer.zero_grad()
            outputs = net(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            # --- Tracking Training Dynamics ---
            _, predicted = outputs.max(1)
            is_clean_mask = (cleans == noisys)
            is_mislabeled_mask = ~is_clean_mask

            # Accuracy on correctly labeled subset
            correct_clean_subset += predicted[is_clean_mask].eq(cleans[is_clean_mask]).sum().item()
            total_clean_subset += is_clean_mask.sum().item()

            # Accuracy on mislabeled subset
            correct_mislabeled_subset += predicted[is_mislabeled_mask].eq(cleans[is_mislabeled_mask]).sum().item()
            total_mislabeled_subset += is_mislabeled_mask.sum().item()

        scheduler.step()

        # --- Validation Tracking ---
        net.eval()
        val_noisy_correct, val_clean_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for _, val_inputs, _, val_cleans, val_noisys in val_loader:
                val_inputs = val_inputs.to(device)
                val_cleans = val_cleans.to(device)
                val_noisys = val_noisys.to(device)

                outputs = net(val_inputs)
                _, predicted = outputs.max(1)

                val_noisy_correct += predicted.eq(val_noisys).sum().item()
                val_clean_correct += predicted.eq(val_cleans).sum().item()
                val_total += val_inputs.size(0)

        # Calculate Epoch Metrics
        train_acc_correct = (correct_clean_subset / total_clean_subset) * 100
        train_acc_mislabeled = (correct_mislabeled_subset / total_mislabeled_subset) * 100
        val_acc_noisy = (val_noisy_correct / val_total) * 100
        val_acc_clean = (val_clean_correct / val_total) * 100

        dynamics_log.append([epoch+1, train_acc_correct, train_acc_mislabeled, val_acc_noisy, val_acc_clean])
        print(f"Epoch {epoch+1:02d} | Train Acc (Clean/Misl): {train_acc_correct:.1f}% / {train_acc_mislabeled:.1f}% | Val Acc (Noisy/Clean): {val_acc_noisy:.1f}% / {val_acc_clean:.1f}%")

    # Save the dynamics table to your predictions folder
    dyn_df = pd.DataFrame(dynamics_log, columns=["epoch", "train_acc_correct", "train_acc_mislabeled", "val_acc_noisy", "val_acc_clean"])
    dyn_df.to_csv("predictions/labelsmoothing_dynamics.csv", index=False)

    return net, val_acc_noisy

# --- Smart Train / Load Block ---
ls_model_path = "models/t4_label_smoothing.pth"
if os.path.exists(ls_model_path):
    print("Found saved Label Smoothing model. Loading...")
    model_ls = ResNet18(num_classes=10).to(device)
    model_ls.load_state_dict(torch.load(ls_model_path, map_location=device))
    # If loading from disk, pull the final accuracy from the saved CSV so the JSON updates correctly
    dyn_df_ls = pd.read_csv("predictions/labelsmoothing_dynamics.csv")
    final_val_acc = float(dyn_df_ls.iloc[-1]['val_acc_noisy'])
else:
    print("Training Task 4 Method 1: Label Smoothing...")
    torch.manual_seed(42)
    model_ls, final_val_acc = train_label_smoothing(train_loader_noisy, val_loader)
    torch.save(model_ls.state_dict(), ls_model_path)

# --- Update JSON automatically ---
with open('results.json', 'r') as f:
    results = json.load(f)

ls_entry = {"name": "LabelSmoothing", "file": "t4_LabelSmoothing.csv", "val_acc_noisy_labels": final_val_acc}

# Filter out the empty template if it exists, otherwise keep existing methods (so we don't overwrite Method 2 later)
if len(results["task4"]["methods"]) > 0 and results["task4"]["methods"][0]["name"] == "":
    results["task4"]["methods"] = [ls_entry]
else:
    # Check if we already logged LabelSmoothing to avoid duplicates
    existing_names = [m["name"] for m in results["task4"]["methods"]]
    if "LabelSmoothing" not in existing_names:
        results["task4"]["methods"].append(ls_entry)

# We will temporarily set this as the best method, but we can programmatically update it later if Method 2 is better
results["task4"]["best_method"] = "LabelSmoothing"

with open('results.json', 'w') as f:
    json.dump(results, f, indent=4)

# --- Generate Predictions ---
print("Generating predictions for Label Smoothing...")
generate_predictions(model_ls, test_loader, "t4_LabelSmoothing.csv")
print("Task 4 - Method 1 completed and saved!")

Found saved Label Smoothing model. Loading...
Generating predictions for Label Smoothing...
Task 4 - Method 1 completed and saved!


In [ ]:
import os
import json
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms # Added import

os.makedirs("models", exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def train_dynamic_relabeling_tta(train_loader, val_loader, epochs=60, warmup_epochs=15, confidence_thresh=0.75):
    net = ResNet18(num_classes=10).to(device)
    criterion = nn.CrossEntropyLoss()

    optimizer = optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # 1. The Fixer Memory Bank
    dynamic_targets = torch.tensor(labels_df['noisy_label'].values, dtype=torch.long).to(device)
    dynamics_log = []

    # NEW: Define the tensor-compatible transforms for TTA
    tta_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip()
    ])

    for epoch in range(epochs):
        net.train()
        labels_changed_this_epoch = 0

        correct_clean_subset, total_clean_subset = 0, 0
        correct_mislabeled_subset, total_mislabeled_subset = 0, 0

        for idx, inputs, _, cleans, noisys in train_loader:
            idx = idx.to(device)
            inputs = inputs.to(device)
            cleans, noisys = cleans.to(device), noisys.to(device)

            # Fetch the current "best known" targets from our memory bank
            current_targets = dynamic_targets[idx]

            optimizer.zero_grad()
            outputs = net(inputs)

            # Train using our dynamically updated targets
            loss = criterion(outputs, current_targets)
            loss.backward()
            optimizer.step()

            # --- 3. AUGMENTATION CONSENSUS RELABELING LOGIC ---
            with torch.no_grad():
                if epoch >= warmup_epochs:
                    # View 1: Use the probability distribution from the current forward pass
                    tta_probs = [F.softmax(outputs, dim=1)]

                    # Views 2 & 3: Generate two more heavily augmented forward passes on the fly
                    for _ in range(2):
                        # NEW: Instantly augment the entire batch of tensors on the GPU!
                        augmented_batch = tta_transform(inputs)
                        view_outputs = net(augmented_batch)
                        tta_probs.append(F.softmax(view_outputs, dim=1))

                    # Average the 3 probability distributions
                    avg_probs = torch.mean(torch.stack(tta_probs), dim=0)
                    max_probs, preds = avg_probs.max(1)

                    # Update Trigger: 75% Consensus Average AND it disagrees with the current target
                    update_mask = (max_probs > confidence_thresh) & (preds != current_targets)

                    if update_mask.any():
                        # Permanently overwrite the dataset labels in memory
                        dynamic_targets[idx[update_mask]] = preds[update_mask]
                        labels_changed_this_epoch += update_mask.sum().item()

            # --- Tracking Training Dynamics ---
            _, predicted = outputs.max(1)
            is_clean_mask = (cleans == noisys)
            is_mislabeled_mask = ~is_clean_mask

            correct_clean_subset += predicted[is_clean_mask].eq(cleans[is_clean_mask]).sum().item()
            total_clean_subset += is_clean_mask.sum().item()
            correct_mislabeled_subset += predicted[is_mislabeled_mask].eq(cleans[is_mislabeled_mask]).sum().item()
            total_mislabeled_subset += is_mislabeled_mask.sum().item()

        scheduler.step()

        # --- Validation Check ---
        net.eval()
        val_noisy_correct, val_clean_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for _, val_inputs, _, val_cleans, val_noisys in val_loader:
                val_inputs = val_inputs.to(device)
                val_cleans = val_cleans.to(device)
                val_noisys = val_noisys.to(device)

                outputs = net(val_inputs)
                _, predicted = outputs.max(1)

                val_noisy_correct += predicted.eq(val_noisys).sum().item()
                val_clean_correct += predicted.eq(val_cleans).sum().item()
                val_total += val_inputs.size(0)

        train_acc_correct = (correct_clean_subset / total_clean_subset) * 100
        train_acc_mislabeled = (correct_mislabeled_subset / total_mislabeled_subset) * 100
        val_acc_noisy = (val_noisy_correct / val_total) * 100
        val_acc_clean = (val_clean_correct / val_total) * 100

        dynamics_log.append([epoch+1, train_acc_correct, train_acc_mislabeled, val_acc_noisy, val_acc_clean])
        print(f"Epoch {epoch+1:02d} | Fixed {labels_changed_this_epoch} bad labels | Val Acc (Noisy): {val_acc_noisy:.1f}%")

    dyn_df = pd.DataFrame(dynamics_log, columns=["epoch", "train_acc_correct", "train_acc_mislabeled", "val_acc_noisy", "val_acc_clean"])
    dyn_df.to_csv("predictions/dynamic_relabeling_tta_dynamics.csv", index=False)

    return net, val_acc_noisy

# --- Smart Train / Load Block ---
tta_model_path = "models/t4_dynamic_relabeling_tta.pth"

if os.path.exists(tta_model_path):
    print("\nFound saved TTA Relabeling model. Loading...")
    model_tta = ResNet18(num_classes=10).to(device)
    model_tta.load_state_dict(torch.load(tta_model_path, map_location=device))

    dyn_df_tta = pd.read_csv("predictions/dynamic_relabeling_tta_dynamics.csv")
    final_val_acc = float(dyn_df_tta.iloc[-1]['val_acc_noisy'])
else:
    print("\nTraining Task 4 Method 2: TTA Dynamic Label Correction...")
    torch.manual_seed(42)
    model_tta, final_val_acc = train_dynamic_relabeling_tta(train_loader_noisy, val_loader)
    torch.save(model_tta.state_dict(), tta_model_path)
    print("TTA Dynamic Relabeling Model saved to Drive.")

# --- Update JSON automatically ---
with open('results.json', 'r') as f:
    results = json.load(f)

tta_entry = {"name": "TTA_Dynamic_Relabeling", "file": "t4_TTA_Dynamic_Relabeling.csv", "val_acc_noisy_labels": final_val_acc}

existing_names = [m["name"] for m in results["task4"]["methods"]]
if "TTA_Dynamic_Relabeling" not in existing_names:
    if len(results["task4"]["methods"]) > 0 and results["task4"]["methods"][0]["name"] == "":
        results["task4"]["methods"] = [tta_entry]
    else:
        results["task4"]["methods"].append(tta_entry)
else:
    for m in results["task4"]["methods"]:
        if m["name"] == "TTA_Dynamic_Relabeling":
            m["val_acc_noisy_labels"] = final_val_acc

best_acc = 0
best_name = ""
for m in results["task4"]["methods"]:
    if m["val_acc_noisy_labels"] is not None and m["val_acc_noisy_labels"] > best_acc:
        best_acc = m["val_acc_noisy_labels"]
        best_name = m["name"]

results["task4"]["best_method"] = best_name

with open('results.json', 'w') as f:
    json.dump(results, f, indent=4)

# --- Generate Predictions ---
print("Generating predictions for TTA Dynamic Relabeling Model...")
generate_predictions(model_tta, test_loader, "t4_TTA_Dynamic_Relabeling.csv")
print("Task 4 - Final Method completed and saved!")


Found saved TTA Relabeling model. Loading...
Generating predictions for TTA Dynamic Relabeling Model...
Task 4 - Final Method completed and saved!


In [ ]:
import os
import json
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from model import ResNet18

os.makedirs("models", exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- 1. The Symmetric Cross Entropy Loss Function ---
class SCELoss(nn.Module): #custome neural network module for it
    def __init__(self, alpha=0.1, beta=1.0, num_classes=10): #alpha dampens , beta maximizes the noise resistance , 10 classes
        super(SCELoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.num_classes = num_classes

    def forward(self, logits, targets):
        # 1. Standard Cross Entropy (CE)
        ce_loss = F.cross_entropy(logits, targets) #standard cross entropy calced

        # 2. Reverse Cross Entropy (RCE)
        # Convert logits to probabilities
        pred = F.softmax(logits, dim=1)
        pred = torch.clamp(pred, min=1e-7, max=1.0) # Prevent log(0) = - infinity

        # Convert targets to one-hot encoding
        targets_one_hot = F.one_hot(targets, num_classes=self.num_classes).float() #Reverse CE requires comparing two probability distributions. This converts the integer target into a 10-slot array of probabilities
        targets_one_hot = torch.clamp(targets_one_hot, min=1e-4, max=1.0) # Prevent log(0)

        # Calculate RCE: The reverse of CE
        rce_loss = (-1 * (pred * torch.log(targets_one_hot)).sum(dim=1)).mean() #it multiplies the network's prediction by the log of the dataset target. , while normal one is  data*log(pred)

        # Combine them using the alpha/beta weighting from the SCE paper
        return self.alpha * ce_loss + self.beta * rce_loss

def train_sce(train_loader, val_loader, epochs=60):
    net = ResNet18(num_classes=10).to(device)

    # We replace the standard nn.CrossEntropyLoss() with our custom SCELoss
    criterion = SCELoss(alpha=0.1, beta=1.0, num_classes=10)

    optimizer = optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    dynamics_log = []

    for epoch in range(epochs):
        net.train()

        correct_clean_subset, total_clean_subset = 0, 0
        correct_mislabeled_subset, total_mislabeled_subset = 0, 0

        # Look how clean this is! No TTA, no dual-networks, no dropped samples.
        for _, inputs, _, cleans, noisys in train_loader:
            inputs = inputs.to(device)
            cleans, noisys = cleans.to(device), noisys.to(device)

            optimizer.zero_grad()
            outputs = net(inputs)

            # The SCE Loss automatically shields the network from the noisy labels
            loss = criterion(outputs, noisys)
            loss.backward()
            optimizer.step()

            # --- TRACKING ---
            with torch.no_grad():
                _, predicted = outputs.max(1)

                is_clean_mask = (cleans == noisys)
                is_mislabeled_mask = ~is_clean_mask

                correct_clean_subset += predicted[is_clean_mask].eq(cleans[is_clean_mask]).sum().item()
                total_clean_subset += is_clean_mask.sum().item()
                correct_mislabeled_subset += predicted[is_mislabeled_mask].eq(cleans[is_mislabeled_mask]).sum().item()
                total_mislabeled_subset += is_mislabeled_mask.sum().item()

        scheduler.step()

        # --- VALIDATION ---
        net.eval()
        val_noisy_correct, val_clean_correct, val_total = 0, 0, 0

        with torch.no_grad():
            for _, val_inputs, _, val_cleans, val_noisys in val_loader:
                val_inputs = val_inputs.to(device)
                val_cleans, val_noisys = val_cleans.to(device), val_noisys.to(device)

                outputs = net(val_inputs)
                _, predicted = outputs.max(1)

                val_noisy_correct += predicted.eq(val_noisys).sum().item()
                val_clean_correct += predicted.eq(val_cleans).sum().item()
                val_total += val_inputs.size(0)

        train_acc_correct = (correct_clean_subset / total_clean_subset) * 100
        train_acc_mislabeled = (correct_mislabeled_subset / total_mislabeled_subset) * 100
        val_acc_noisy = (val_noisy_correct / val_total) * 100
        val_acc_clean = (val_clean_correct / val_total) * 100

        dynamics_log.append([epoch+1, train_acc_correct, train_acc_mislabeled, val_acc_noisy, val_acc_clean])
        print(f"Epoch {epoch+1:02d} | Train Acc (Mislabeled): {train_acc_mislabeled:.1f}% | Val Acc (Noisy): {val_acc_noisy:.1f}%")

    dyn_df = pd.DataFrame(dynamics_log, columns=["epoch", "train_acc_correct", "train_acc_mislabeled", "val_acc_noisy", "val_acc_clean"])
    dyn_df.to_csv("predictions/sce_dynamics.csv", index=False)

    return net, val_acc_noisy

# --- Smart Train / Load Block ---
sce_model_path = "models/t4_sce.pth"

if os.path.exists(sce_model_path):
    print("\nFound saved SCE model. Loading...")
    model_sce = ResNet18(num_classes=10).to(device)
    model_sce.load_state_dict(torch.load(sce_model_path, map_location=device))

    dyn_df_sce = pd.read_csv("predictions/sce_dynamics.csv")
    final_val_acc = float(dyn_df_sce.iloc[-1]['val_acc_noisy'])
else:
    print("\nTraining Task 4 Method 3: Symmetric Cross Entropy (SCE)...")
    torch.manual_seed(42)
    model_sce, final_val_acc = train_sce(train_loader_noisy, val_loader)
    torch.save(model_sce.state_dict(), sce_model_path)
    print("SCE Model saved to Drive.")

# --- Update JSON automatically ---
with open('results.json', 'r') as f:
    results = json.load(f)

sce_entry = {"name": "Symmetric_Cross_Entropy", "file": "t4_SCE.csv", "val_acc_noisy_labels": final_val_acc}

existing_names = [m["name"] for m in results["task4"]["methods"]]
if "Symmetric_Cross_Entropy" not in existing_names:
    results["task4"]["methods"].append(sce_entry)
else:
    for m in results["task4"]["methods"]:
        if m["name"] == "Symmetric_Cross_Entropy":
            m["val_acc_noisy_labels"] = final_val_acc

best_acc = 0
best_name = ""
for m in results["task4"]["methods"]:
    if m["val_acc_noisy_labels"] is not None and m["val_acc_noisy_labels"] > best_acc:
        best_acc = m["val_acc_noisy_labels"]
        best_name = m["name"]

results["task4"]["best_method"] = best_name

with open('results.json', 'w') as f:
    json.dump(results, f, indent=4)

# --- Generate Predictions ---
print("Generating predictions for SCE Model...")
generate_predictions(model_sce, test_loader, "t4_SCE.csv")
print("Task 4 - Final Method completed and saved!")


Found saved SCE model. Loading...
Generating predictions for SCE Model...
Task 4 - Final Method completed and saved!


In [ ]:
import os
import json
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from model import ResNet18

os.makedirs("models", exist_ok=True)
os.makedirs("predictions", exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load the synthetic labels into a master GPU tensor
synthetic_master_targets = torch.tensor(labels_df['synthetic_label'].values, dtype=torch.long).to(device)

# --- 1. Task 5: Baseline Cross-Entropy (Synthetic Noise) ---
def train_task5_baseline(train_loader, val_loader, epochs=60):
    net = ResNet18(num_classes=10).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    dynamics_log = []

    for epoch in range(epochs):
        net.train()
        correct_clean_subset, total_clean_subset = 0, 0
        correct_mislabeled_subset, total_mislabeled_subset = 0, 0

        # We ignore the 'noisys' from the dataloader and inject the synthetic ones
        for idx, inputs, _, cleans, _ in train_loader:
            idx = idx.to(device)
            inputs = inputs.to(device)
            cleans = cleans.to(device)

            # INJECT SYNTHETIC NOISE TARGETS HERE
            synth_targets = synthetic_master_targets[idx]

            optimizer.zero_grad()
            outputs = net(inputs)
            loss = criterion(outputs, synth_targets)
            loss.backward()
            optimizer.step()

            # --- TRACKING ---
            with torch.no_grad():
                _, predicted = outputs.max(1)

                # Check memorization against the SYNTHETIC mask
                is_clean_mask = (cleans == synth_targets)
                is_mislabeled_mask = ~is_clean_mask

                correct_clean_subset += predicted[is_clean_mask].eq(cleans[is_clean_mask]).sum().item()
                total_clean_subset += is_clean_mask.sum().item()
                correct_mislabeled_subset += predicted[is_mislabeled_mask].eq(cleans[is_mislabeled_mask]).sum().item()
                total_mislabeled_subset += is_mislabeled_mask.sum().item()

        scheduler.step()

        # --- VALIDATION ---
        net.eval()
        val_clean_correct, val_total = 0, 0
        with torch.no_grad():
            for _, val_inputs, _, val_cleans, _ in val_loader:
                val_inputs, val_cleans = val_inputs.to(device), val_cleans.to(device)
                outputs = net(val_inputs)
                _, predicted = outputs.max(1)

                val_clean_correct += predicted.eq(val_cleans).sum().item()
                val_total += val_inputs.size(0)

        train_acc_mislabeled = (correct_mislabeled_subset / total_mislabeled_subset) * 100 if total_mislabeled_subset > 0 else 0
        val_acc_clean = (val_clean_correct / val_total) * 100

        dynamics_log.append([epoch+1, train_acc_mislabeled, val_acc_clean])
        print(f"[Baseline] Epoch {epoch+1:02d} | Train Acc (Mislabeled): {train_acc_mislabeled:.1f}% | Val Acc (Clean): {val_acc_clean:.1f}%")

    dyn_df = pd.DataFrame(dynamics_log, columns=["epoch", "train_acc_mislabeled", "val_acc_clean"])
    dyn_df.to_csv("predictions/t5_baseline_synthetic_dynamics.csv", index=False)
    return net, val_acc_clean

# --- 2. Task 5: Pure SCE (Synthetic Noise) ---
class SCELoss(nn.Module):
    def __init__(self, alpha=0.1, beta=1.0, num_classes=10):
        super(SCELoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.num_classes = num_classes

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets)
        pred = F.softmax(logits, dim=1)
        pred = torch.clamp(pred, min=1e-7, max=1.0)
        targets_one_hot = F.one_hot(targets, num_classes=self.num_classes).float()
        targets_one_hot = torch.clamp(targets_one_hot, min=1e-4, max=1.0)
        rce_loss = (-1 * (pred * torch.log(targets_one_hot)).sum(dim=1)).mean()
        return self.alpha * ce_loss + self.beta * rce_loss

def train_task5_sce(train_loader, val_loader, epochs=60):
    net = ResNet18(num_classes=10).to(device)
    criterion = SCELoss(alpha=0.1, beta=1.0, num_classes=10)
    optimizer = optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    dynamics_log = []

    for epoch in range(epochs):
        net.train()
        correct_clean_subset, total_clean_subset = 0, 0
        correct_mislabeled_subset, total_mislabeled_subset = 0, 0

        for idx, inputs, _, cleans, _ in train_loader:
            idx = idx.to(device)
            inputs = inputs.to(device)
            cleans = cleans.to(device)
            synth_targets = synthetic_master_targets[idx]

            optimizer.zero_grad()
            outputs = net(inputs)
            loss = criterion(outputs, synth_targets)
            loss.backward()
            optimizer.step()

            with torch.no_grad():
                _, predicted = outputs.max(1)
                is_clean_mask = (cleans == synth_targets)
                is_mislabeled_mask = ~is_clean_mask

                correct_clean_subset += predicted[is_clean_mask].eq(cleans[is_clean_mask]).sum().item()
                total_clean_subset += is_clean_mask.sum().item()
                correct_mislabeled_subset += predicted[is_mislabeled_mask].eq(cleans[is_mislabeled_mask]).sum().item()
                total_mislabeled_subset += is_mislabeled_mask.sum().item()

        scheduler.step()

        net.eval()
        val_clean_correct, val_total = 0, 0
        with torch.no_grad():
            for _, val_inputs, _, val_cleans, _ in val_loader:
                val_inputs, val_cleans = val_inputs.to(device), val_cleans.to(device)
                outputs = net(val_inputs)
                _, predicted = outputs.max(1)

                val_clean_correct += predicted.eq(val_cleans).sum().item()
                val_total += val_inputs.size(0)

        train_acc_mislabeled = (correct_mislabeled_subset / total_mislabeled_subset) * 100 if total_mislabeled_subset > 0 else 0
        val_acc_clean = (val_clean_correct / val_total) * 100

        dynamics_log.append([epoch+1, train_acc_mislabeled, val_acc_clean])
        print(f"[SCE] Epoch {epoch+1:02d} | Train Acc (Mislabeled): {train_acc_mislabeled:.1f}% | Val Acc (Clean): {val_acc_clean:.1f}%")

    dyn_df = pd.DataFrame(dynamics_log, columns=["epoch", "train_acc_mislabeled", "val_acc_clean"])
    dyn_df.to_csv("predictions/t5_sce_synthetic_dynamics.csv", index=False)
    return net, val_acc_clean

# --- Smart Execution & Saving Block ---
t5_base_path = "models/t5_baseline_synthetic.pth"
t5_base_csv = "predictions/t5_baseline_synthetic_dynamics.csv"

if os.path.exists(t5_base_path) and os.path.exists(t5_base_csv):
    print("\nFound saved Task 5 Baseline model. Loading...")
    model_t5_base = ResNet18(num_classes=10).to(device)
    model_t5_base.load_state_dict(torch.load(t5_base_path, map_location=device))
    val_acc_base = float(pd.read_csv(t5_base_csv).iloc[-1]['val_acc_clean'])
else:
    print("\n--- Training Task 5 Baseline (Synthetic) ---")
    torch.manual_seed(42)
    model_t5_base, val_acc_base = train_task5_baseline(train_loader_noisy, val_loader)
    torch.save(model_t5_base.state_dict(), t5_base_path)
    print("Task 5 Baseline weights saved.")

t5_sce_path = "models/t5_sce_synthetic.pth"
t5_sce_csv = "predictions/t5_sce_synthetic_dynamics.csv"

if os.path.exists(t5_sce_path) and os.path.exists(t5_sce_csv):
    print("\nFound saved Task 5 Pure SCE model. Loading...")
    model_t5_sce = ResNet18(num_classes=10).to(device)
    model_t5_sce.load_state_dict(torch.load(t5_sce_path, map_location=device))
    val_acc_sce = float(pd.read_csv(t5_sce_csv).iloc[-1]['val_acc_clean'])
else:
    print("\n--- Training Task 5 Pure SCE (Synthetic) ---")
    torch.manual_seed(42)
    model_t5_sce, val_acc_sce = train_task5_sce(train_loader_noisy, val_loader)
    torch.save(model_t5_sce.state_dict(), t5_sce_path)
    print("Task 5 Pure SCE weights saved.")

# --- Update JSON ---
if os.path.exists('results.json'):
    with open('results.json', 'r') as f:
        results = json.load(f)
else:
    results = {}

# Ensure task5 dictionary exists
if "task5" not in results:
    results["task5"] = {}

# Update the exact keys required by the evaluation rubric
results["task5"]["val_acc_synthetic_labels__baseline"] = val_acc_base
results["task5"]["val_acc_synthetic_labels__best_method"] = val_acc_sce

# Remove the incorrect "methods" array if it was generated by the previous run
if "methods" in results["task5"]:
    del results["task5"]["methods"]

with open('results.json', 'w') as f:
    json.dump(results, f, indent=4)

print("\nresults.json has been correctly updated to match the required format!")

# --- Generate Task 5 Predictions ---
print("Generating predictions for Task 5...")
generate_predictions(model_t5_base, test_loader, "t5_baseline_synth.csv")
generate_predictions(model_t5_sce, test_loader, "t5_best_synth.csv")


Found saved Task 5 Baseline model. Loading...

Found saved Task 5 Pure SCE model. Loading...

results.json has been correctly updated to match the required format!


In [ ]:
#Bad Task4 method dont bother with it, just use the ones from the JSON 






import os
import json
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
from model import ResNet18

os.makedirs("models", exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- 1. The Symmetric Cross Entropy Loss Function (The Shield) ---
class SCELoss(nn.Module):
    def __init__(self, alpha=0.1, beta=1.0, num_classes=10):
        super(SCELoss, self).__init__() #calling super() runs all the foundational setup code hidden inside nn.Module before applying your custom logic
        self.alpha = alpha
        self.beta = beta
        self.num_classes = num_classes

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets)

        pred = F.softmax(logits, dim=1)
        pred = torch.clamp(pred, min=1e-7, max=1.0)

        targets_one_hot = F.one_hot(targets, num_classes=self.num_classes).float()
        targets_one_hot = torch.clamp(targets_one_hot, min=1e-4, max=1.0)

        rce_loss = (-1 * (pred * torch.log(targets_one_hot)).sum(dim=1)).mean()
        return self.alpha * ce_loss + self.beta * rce_loss

# --- 2. The Hybrid Training Loop (Shield + Fixer) ---
def train_hybrid_sce_tta(train_loader, val_loader, epochs=60, warmup_epochs=15, confidence_thresh=0.75):
    net = ResNet18(num_classes=10).to(device)

    # We replace standard CrossEntropy with SCE
    criterion = SCELoss(alpha=0.1, beta=1.0, num_classes=10)

    optimizer = optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # The Fixer Memory Bank
    dynamic_targets = torch.tensor(labels_df['noisy_label'].values, dtype=torch.long).to(device)
    dynamics_log = []

    # GPU Tensor Transforms for TTA
    tta_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip()
    ])

    for epoch in range(epochs):
        net.train()
        labels_changed_this_epoch = 0

        correct_clean_subset, total_clean_subset = 0, 0
        correct_mislabeled_subset, total_mislabeled_subset = 0, 0

        for idx, inputs, _, cleans, noisys in train_loader:
            idx = idx.to(device)
            inputs = inputs.to(device)
            cleans, noisys = cleans.to(device), noisys.to(device)

            # Fetch the current "best known" targets from our memory bank
            current_targets = dynamic_targets[idx]

            optimizer.zero_grad()
            outputs = net(inputs)

            # Train using our dynamically updated targets WITH the SCE mathematical shield!
            loss = criterion(outputs, current_targets)
            loss.backward()
            optimizer.step()

            # --- 3. AUGMENTATION CONSENSUS RELABELING LOGIC ---
            with torch.no_grad():
                if epoch >= warmup_epochs:
                    tta_probs = [F.softmax(outputs, dim=1)]

                    for _ in range(2):
                        augmented_batch = tta_transform(inputs)
                        view_outputs = net(augmented_batch)
                        tta_probs.append(F.softmax(view_outputs, dim=1))

                    avg_probs = torch.mean(torch.stack(tta_probs), dim=0)
                    max_probs, preds = avg_probs.max(1)

                    update_mask = (max_probs > confidence_thresh) & (preds != current_targets)

                    if update_mask.any():
                        dynamic_targets[idx[update_mask]] = preds[update_mask]
                        labels_changed_this_epoch += update_mask.sum().item()

            # --- Tracking Training Dynamics ---
            _, predicted = outputs.max(1)
            is_clean_mask = (cleans == noisys)
            is_mislabeled_mask = ~is_clean_mask

            correct_clean_subset += predicted[is_clean_mask].eq(cleans[is_clean_mask]).sum().item()
            total_clean_subset += is_clean_mask.sum().item()
            correct_mislabeled_subset += predicted[is_mislabeled_mask].eq(cleans[is_mislabeled_mask]).sum().item()
            total_mislabeled_subset += is_mislabeled_mask.sum().item()

        scheduler.step()

        # --- Validation Check ---
        net.eval()
        val_noisy_correct, val_clean_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for _, val_inputs, _, val_cleans, val_noisys in val_loader:
                val_inputs = val_inputs.to(device)
                val_cleans, val_noisys = val_cleans.to(device), val_noisys.to(device)

                outputs = net(val_inputs)
                _, predicted = outputs.max(1)

                val_noisy_correct += predicted.eq(val_noisys).sum().item()
                val_clean_correct += predicted.eq(val_cleans).sum().item()
                val_total += val_inputs.size(0)

        train_acc_correct = (correct_clean_subset / total_clean_subset) * 100
        train_acc_mislabeled = (correct_mislabeled_subset / total_mislabeled_subset) * 100
        val_acc_noisy = (val_noisy_correct / val_total) * 100
        val_acc_clean = (val_clean_correct / val_total) * 100

        dynamics_log.append([epoch+1, train_acc_correct, train_acc_mislabeled, val_acc_noisy, val_acc_clean])
        print(f"Epoch {epoch+1:02d} | Fixed {labels_changed_this_epoch} bad labels | Val Acc (Noisy): {val_acc_noisy:.1f}%")

    dyn_df = pd.DataFrame(dynamics_log, columns=["epoch", "train_acc_correct", "train_acc_mislabeled", "val_acc_noisy", "val_acc_clean"])
    dyn_df.to_csv("predictions/t4_hybrid_sce_tta_dynamics.csv", index=False)

    return net, val_acc_noisy

# --- Smart Train / Load Block ---
hybrid_model_path = "models/t4_hybrid_sce_tta.pth"

if os.path.exists(hybrid_model_path):
    print("\nFound saved Hybrid SCE+TTA model. Loading...")
    model_hybrid = ResNet18(num_classes=10).to(device)
    model_hybrid.load_state_dict(torch.load(hybrid_model_path, map_location=device))

    dyn_df_hybrid = pd.read_csv("predictions/t4_hybrid_sce_tta_dynamics.csv")
    final_val_acc = float(dyn_df_hybrid.iloc[-1]['val_acc_noisy'])
else:
    print("\nTraining Task 4 Method 3: Hybrid (SCE Loss + TTA Relabeling)...")
    torch.manual_seed(42)
    model_hybrid, final_val_acc = train_hybrid_sce_tta(train_loader_noisy, val_loader)
    torch.save(model_hybrid.state_dict(), hybrid_model_path)
    print("Hybrid Model saved to Drive.")

# --- Update JSON automatically ---
with open('results.json', 'r') as f:
    results = json.load(f)

hybrid_entry = {"name": "Hybrid_SCE_TTA", "file": "t4_Hybrid_SCE_TTA.csv", "val_acc_noisy_labels": final_val_acc}

existing_names = [m["name"] for m in results["task4"]["methods"]]
if "Hybrid_SCE_TTA" not in existing_names:
    if len(results["task4"]["methods"]) > 0 and results["task4"]["methods"][0]["name"] == "":
        results["task4"]["methods"] = [hybrid_entry]
    else:
        results["task4"]["methods"].append(hybrid_entry)
else:
    for m in results["task4"]["methods"]:
        if m["name"] == "Hybrid_SCE_TTA":
            m["val_acc_noisy_labels"] = final_val_acc

best_acc = 0
best_name = ""
for m in results["task4"]["methods"]:
    if m["val_acc_noisy_labels"] is not None and m["val_acc_noisy_labels"] > best_acc:
        best_acc = m["val_acc_noisy_labels"]
        best_name = m["name"]

with open('results.json', 'w') as f:
    json.dump(results, f, indent=4)

# --- Generate Predictions ---
print("Generating predictions for Hybrid SCE+TTA Model...")
generate_predictions(model_hybrid, test_loader, "t4_Hybrid_SCE_TTA.csv")
print("Task 4 - Final Hybrid Method completed and saved!")

# FOLLOWING WAS THE OUTPUT UPON RUNNING THE ABOVE MODEL FOR TASK 4,

DUE TO THE POOR PERFORMANCE I DID NOT CONSIDER THIS AS ONE OF MY 3 MODELS ,
THIS PERFORMANCE WAS UNEXPECTED AS IT MERGED 2 GREAT IDEAS BUT MADE SENSE WHEN LOOKED DEEP INTO HOW THE CODES MERGE

Training Task 4 Method 3: Hybrid (SCE Loss + TTA Relabeling)...

Epoch 01 | Fixed 0 bad labels | Val Acc (Noisy): 24.8%

Epoch 02 | Fixed 0 bad labels | Val Acc (Noisy): 25.7%

Epoch 03 | Fixed 0 bad labels | Val Acc (Noisy): 30.3%

Epoch 04 | Fixed 0 bad labels | Val Acc (Noisy): 31.7%

Epoch 05 | Fixed 0 bad labels | Val Acc (Noisy): 33.6%

Epoch 06 | Fixed 0 bad labels | Val Acc (Noisy): 35.7%

Epoch 07 | Fixed 0 bad labels | Val Acc (Noisy): 36.9%

Epoch 08 | Fixed 0 bad labels | Val Acc (Noisy): 39.3%

Epoch 09 | Fixed 0 bad labels | Val Acc (Noisy): 33.9%

Epoch 10 | Fixed 0 bad labels | Val Acc (Noisy): 40.7%

Epoch 11 | Fixed 0 bad labels | Val Acc (Noisy): 43.5%

Epoch 12 | Fixed 0 bad labels | Val Acc (Noisy): 41.0%

Epoch 13 | Fixed 0 bad labels | Val Acc (Noisy): 41.8%

Epoch 14 | Fixed 0 bad labels | Val Acc (Noisy): 43.2%

Epoch 15 | Fixed 0 bad labels | Val Acc (Noisy): 41.0%

Epoch 16 | Fixed 12563 bad labels | Val Acc (Noisy): 46.9%

Epoch 17 | Fixed 7573 bad labels | Val Acc (Noisy): 44.6%

Epoch 18 | Fixed 5950 bad labels | Val Acc (Noisy): 46.6%

Epoch 19 | Fixed 5236 bad labels | Val Acc (Noisy): 45.7%

Epoch 20 | Fixed 4876 bad labels | Val Acc (Noisy): 43.8%

Epoch 21 | Fixed 4635 bad labels | Val Acc (Noisy): 45.2%

Epoch 22 | Fixed 4474 bad labels | Val Acc (Noisy): 45.0%

Epoch 23 | Fixed 4362 bad labels | Val Acc (Noisy): 43.7%

Epoch 24 | Fixed 4201 bad labels | Val Acc (Noisy): 43.8%
\
Epoch 25 | Fixed 4122 bad labels | Val Acc (Noisy): 44.2%

Epoch 26 | Fixed 3884 bad labels | Val Acc (Noisy): 41.7%

Epoch 27 | Fixed 3978 bad labels | Val Acc (Noisy): 41.7%

Epoch 28 | Fixed 3833 bad labels | Val Acc (Noisy): 42.9%

Epoch 29 | Fixed 3675 bad labels | Val Acc (Noisy): 41.6%

Epoch 30 | Fixed 3453 bad labels | Val Acc (Noisy): 41.7%

Epoch 31 | Fixed 3315 bad labels | Val Acc (Noisy): 41.5%

Epoch 32 | Fixed 3399 bad labels | Val Acc (Noisy): 40.8%

Epoch 33 | Fixed 3362 bad labels | Val Acc (Noisy): 39.5%

Epoch 34 | Fixed 3337 bad labels | Val Acc (Noisy): 38.6%

Epoch 35 | Fixed 3235 bad labels | Val Acc (Noisy): 40.3%

Epoch 36 | Fixed 3162 bad labels | Val Acc (Noisy): 39.2%

Epoch 37 | Fixed 3056 bad labels | Val Acc (Noisy): 39.3%

Epoch 38 | Fixed 2978 bad labels | Val Acc (Noisy): 39.1%

Epoch 39 | Fixed 2829 bad labels | Val Acc (Noisy): 38.8%

Epoch 40 | Fixed 2713 bad labels | Val Acc (Noisy): 38.6%


After this i terminated it.